# Python Basics — Weekend 2 Reference
**Course:** Python Applications with OpenAI — MSME Technology Development Center, Chennai  
**Type:** Supplementary reference notebook (use alongside Sessions 3 & 4)

---

## Purpose
Sessions 3 and 4 introduce tool calling, structured outputs, streaming, and multimodal APIs. All of these lean heavily on specific Python patterns: **dicts**, **functions with `**kwargs`**, **list comprehensions**, **try/except**, **classes**, and **JSON handling**.

This notebook is a **hands-on refresher** on exactly those patterns — no OpenAI API needed. Run it any time you hit a Python concept you want to review.

## Topics
1. Strings, numbers, and f-strings
2. Lists and dicts in depth
3. Comprehensions (list, dict)
4. Control flow: `if`, `for`, `while`
5. Functions: defaults, `*args`, `**kwargs`, type hints
6. Lambda and closures
7. Error handling: `try / except / finally`
8. Classes and objects
9. JSON: `json.loads`, `json.dumps`
10. File I/O with `pathlib`

In [1]:
# ── Setup (no OpenAI key required for this notebook) ─────────────────────────
import json
import os
from pathlib import Path

print("Python", __import__('sys').version.split()[0])
print("Ready.")

Python 3.12.3
Ready.


---
## 1. Strings, Numbers, and f-strings

In AI apps you will constantly format model prompts and parse replies. **f-strings** are the cleanest way to embed variables into strings. Key behaviours:
- `f"{value}"` — basic interpolation
- `f"{value:.2f}"` — format spec (2 decimal places)
- `f"{value!r}"` — `repr()` form (shows quotes around strings — useful for debugging)
- Multi-line f-strings work with triple quotes

In [5]:
# datatypes in python

# string 

fruit = 'apple'

seasonal_fruit = "Mango"

prompt = """
You are an intelligent customer support agent.
You work with tools available to get information live from the databases using
...
...

"""


product = 'headphone for workout'


prompt_product_name =  f"""
Suggest a good product name for product  given below.
The name should be trendy and catchy.

product : {product}

"""

# float & integer - numbers

pi = 3.14

num_of_oranges = 6

radius_of_orange = 12


circumference = 2*pi*radius_of_orange

print(circumference)



# boolean (logical operations AND/OR )

flag_available = True

check_passed = False

75.36


In [ ]:
# ── 1a. f-string basics ──────────────────────────────────────────────────────
# STEPS:
#   1. name = "Priya"  |  tokens = 1234  |  cost = 0.000185
#   2. Print: "User: Priya  |  Tokens used: 1,234  |  Cost: $0.000185"
#      using f"{tokens:,}" for comma-separated thousands
#      and f"{cost:.6f}" for 6 decimal places
#   3. Build a multi-line prompt string with triple-quote f-string:
#      role = "financial advisor"
#      domain = "GST filing"
#      system_prompt = f"""You are a {role}. ..."""
#      print(repr(system_prompt))   # !r shows \n as literal \n
# --- live-code below ---





In [ ]:
# ── 1b. Useful string methods for AI work ────────────────────────────────────
# STEPS:
#   1. reply = "  The answer is: 42. \n"
#      print(reply.strip())          # remove whitespace — do this on every API reply
#      print(reply.strip().lower())  # normalise for comparison
#
#   2. CSV-style output from a model:
#      csv_reply = "Mumbai, Chennai, Delhi, Bangalore"
#      cities = [c.strip() for c in csv_reply.split(",")]
#      print(cities)                 # ['Mumbai', 'Chennai', 'Delhi', 'Bangalore']
#
#   3. Check if a reply contains a keyword (case-insensitive):
#      if "error" in reply.lower():
#          print("Reply mentions an error")
#      else:
#          print("Looks fine")
# --- live-code below ---


reply = " The answer is: 42. \n Text end"


print(reply)

print(reply.strip())

print(reply.upper())

 The answer is: 42. 
 Text end
The answer is: 42. 
 Text end
 THE ANSWER IS: 42. 
 TEXT END


In [12]:
# logical operations

print(True and False)

print(True and True)

print(True or False )

print(False and False)


False
True
True
False


---
## 2. Lists and Dicts In Depth

Every OpenAI API request is a **dict**; the `messages` parameter is a **list of dicts**. Being fluent with these two types is the single most important Python skill for this course.

### Dict cheat-sheet
```python
d = {"role": "user", "content": "Hello"}   # literal
d["role"]               # access — raises KeyError if missing
d.get("role", "user")   # safe access with default
d["new_key"] = "value"  # add / update
del d["new_key"]        # remove
"role" in d             # membership test
d.keys()  d.values()  d.items()   # iteration views
{**d, "extra": 1}       # merge / spread (Python 3.9+: d | {"extra": 1})
```

In [ ]:
# list

fruits = ['apple','banana','papaya','mango']



# tuple

criterias = ('amount gte 20','volume != 0')


# dictionary

cities = {'tn':"chennai",'ap':'hyderabad','delhi':"delhi"}


In [ ]:
fruits[2]



'papaya'

In [15]:
criterias[1]

'volume != 0'

In [16]:
cities['ap']

'hyderabad'

In [17]:
# ── 2a. Build and manipulate the messages list ───────────────────────────────
# STEPS:
#   1. Create the messages list that you would send to the API:
#      messages = [
#        {"role": "system",    "content": "You are a helpful assistant."},
#        {"role": "user",      "content": "What is Python?"},
#        {"role": "assistant", "content": "Python is a programming language."},
#        {"role": "user",      "content": "Name one library for data science."},
#      ]
#
#   2. Print only the user messages:
#      user_msgs = [m["content"] for m in messages if m["role"] == "user"]
#      print(user_msgs)
#
#   3. Count tokens (rough estimate: 4 chars ≈ 1 token):
#      total_chars = sum(len(m["content"]) for m in messages)
#      print(f"Rough token estimate: {total_chars // 4}")
#
#   4. Append a new turn and print the length of messages:
#      messages.append({"role": "assistant", "content": "NumPy."})
#      print(f"History length: {len(messages)} messages")
# --- live-code below ---

messages = [
       {"role": "system",    "content": "You are a helpful assistant."},
       {"role": "user",      "content": "What is Python?"},
       {"role": "assistant", "content": "Python is a programming language."},
       {"role": "user",      "content": "Name one library for data science."},
     ]


print(messages[2])


{'role': 'assistant', 'content': 'Python is a programming language.'}


In [ ]:
# ── 2b. Nested dicts — like an API response object ───────────────────────────
# STEPS:
#   1. Simulate a simplified API response as a plain dict:
#      response = {
#        "id": "chatcmpl-abc123",
#        "model": "gpt-5.4-mini",
#        "choices": [
#          {
#            "index": 0,
#            "message": {"role": "assistant", "content": "NumPy is great for arrays."},
#            "finish_reason": "stop"
#          }
#        ],
#        "usage": {"prompt_tokens": 25, "completion_tokens": 8, "total_tokens": 33}
#      }
#
#   2. Access the reply text:    response["choices"][0]["message"]["content"]
#   3. Access finish reason:     response["choices"][0]["finish_reason"]
#   4. Access total tokens:      response["usage"]["total_tokens"]
#   5. Use .get() to safely read a field that might not exist:
#      system_fp = response.get("system_fingerprint", "not available")
#      print(system_fp)
# --- live-code below ---


---
## 3. Comprehensions

Comprehensions replace `for` loops that build a new list or dict — they are faster to read and write.

```python
# List comprehension
[expression for item in iterable if condition]

# Dict comprehension
{key_expr: value_expr for item in iterable if condition}
```

In [ ]:
# ── 3a. List and dict comprehensions on API data ─────────────────────────────
# STEPS:
#   1. Given a list of tool calls (simulated):
#      calls = [
#        {"name": "get_usd_to_inr",  "args": {}},
#        {"name": "calculate_gst",    "args": {"amount": 10000, "gst_rate": 18}},
#        {"name": "get_usd_to_inr",  "args": {}},
#      ]
#
#   2. List of unique tool names called:
#      unique_names = list({c["name"] for c in calls})  # set comprehension → no duplicates
#      print(unique_names)
#
#   3. Filter only calls that have arguments:
#      calls_with_args = [c for c in calls if c["args"]]
#      print(calls_with_args)
#
#   4. Dict mapping name → count:
#      from collections import Counter
#      counts = dict(Counter(c["name"] for c in calls))
#      print(counts)   # {"get_usd_to_inr": 2, "calculate_gst": 1}
#
#   5. Extract just the "amount" from each call that has it:
#      amounts = [c["args"]["amount"] for c in calls if "amount" in c["args"]]
#      print(amounts)
# --- live-code below ---


---
## 4. Control Flow

Two patterns from Session 3 you should be fluent with:
- **`if / elif / else`** — dispatching based on `finish_reason` or exception type
- **`for` loops** — iterating over messages, stream chunks, tool calls

In [20]:
# ── 4a. if/elif/else — dispatching on finish_reason ──────────────────────────
# STEPS:
#   1. finish_reasons = ["stop", "length", "tool_calls", "content_filter", "stop"]
#
#   2. For each reason, write an if/elif/else that prints what to do:
#      "stop"           → "Reply complete — extract text"
#      "tool_calls"     → "Execute tool and send result back"
#      "length"         → "Reply truncated — increase max_tokens or trim history"
#      "content_filter" → "Content blocked — adjust prompt"
#      else             → f"Unknown finish reason: {reason}"
#
#   3. Print the action for each reason in the list
# --- live-code below ---


number = 24


if number%2 == 0:

    print('This is an even number')

elif number%2 == 1:

    print('This is an odd number')

else:

    print('Unable to find the operation')


This is an even number


In [22]:
# ── 4b. for loops — iterating stream chunks and messages ─────────────────────
# STEPS:
#   1. Simulate a stream as a list of chunks (each has a delta string or None):
#      chunks = [
#        {"delta": "NumPy"},
#        {"delta": " is"},
#        {"delta": " great"},
#        {"delta": None},    # None means the stream ended
#        {"delta": "."},
#      ]
#
#   2. Reconstruct full text (skipping None deltas):
#      full = ""
#      for chunk in chunks:
#          if chunk["delta"] is not None:
#              full += chunk["delta"]
#              print(chunk["delta"], end="", flush=True)
#      print()  # newline
#      print("Full text:", full)
#
#   3. Using enumerate to track turn numbers:
#      messages = [{"role": "user", "content": "Q1"}, {"role": "assistant", "content": "A1"}]
#      for i, msg in enumerate(messages, 1):
#          print(f"  [{i}] {msg['role']}: {msg['content']}")
# --- live-code below ---


# for loop

for num in range(50):

    print(num)



# while loop
num = 0

while num < 50:

    print(num)

    num = num+1




0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49


---
## 5. Functions: Defaults, `*args`, `**kwargs`, Type Hints

Most helpers in `utils/` and the patterns in Session 3 use these:

```python
def fn(required, optional="default", *args, **kwargs):
    ...
```

- `*args` — collects extra positional arguments into a **tuple**
- `**kwargs` — collects extra keyword arguments into a **dict**
- **Type hints** don't enforce types at runtime but make code readable and support IDE autocompletion

```python
def chat(prompt: str, model: str = "gpt-5.4-mini", **kwargs) -> str:
    ...
```

In [23]:
# ── 5a. Functions with defaults and type hints ───────────────────────────────
# STEPS:
#   1. Write build_messages(user: str, system: str = "You are a helpful assistant.") -> list:
#      Returns [{"role": "system", "content": system}, {"role": "user", "content": user}]
#
#   2. Call it two ways:
#      a) build_messages("What is GST?")                           # uses default system
#      b) build_messages("What is GST?", system="You are a CA.")   # overrides system
#      Print both results
#
#   3. Add a return type annotation and verify with:
#      import inspect
#      print(inspect.signature(build_messages))
# --- live-code below ---


def calculate_area_of_square(length):
    

    return length * length
    


In [24]:
calculate_area_of_square(length=25)

625

In [25]:
def travel_planer(start_point,destination,num_of_days=10,budget=100000):

    # docstring
    """
    Uses openai llm model and builds a comprehensive travel iternary
    """

    travel_itenary =  ''

    # open llm call

    # parse the reponse


    return travel_itenary



In [26]:
travel_planer(start_point='chennai',destination='madurai',budget=10000)

''

In [ ]:
# ── 5b. **kwargs — passing arbitrary API parameters through ─────────────────
# This pattern appears in chat_turn() and safe_chat() from Session 3.
#
# STEPS:
#   1. def call_api(messages: list, model: str, **kwargs):
#        """kwargs are forwarded straight to create()"""
#        print(f"model={model}, extra params={kwargs}")
#        # In real code: client.chat.completions.create(model=model, messages=messages, **kwargs)
#
#   2. Call it with different combos:
#      call_api([], "gpt-5.4-mini")
#      call_api([], "gpt-5.4-mini", temperature=0.3)
#      call_api([], "gpt-5.4-mini", temperature=0.7, max_tokens=100, stop=["."])
#
#   3. Inside the function, inspect kwargs:
#      for key, val in kwargs.items():
#          print(f"  {key} = {val!r}")
# --- live-code below ---


---
## 6. Lambda and Closures

The `retry()` helper in `utils/helpers.py` takes a **callable** — a function with no arguments. The standard pattern is to wrap the API call in a `lambda`:

```python
response = retry(
    lambda: client.chat.completions.create(model=MODEL, messages=messages)
)
```

A **closure** is a function that captures variables from the enclosing scope — `lambda` naturally creates closures.

In [ ]:
# ── 6a. Lambda and the callable pattern used in retry() ─────────────────────
# STEPS:
#   1. Simple lambda:
#      double = lambda x: x * 2
#      print(double(5))   # 10
#
#   2. Lambda as a closure — captures outer variable:
#      multiplier = 3
#      triple = lambda x: x * multiplier
#      print(triple(5))   # 15
#      multiplier = 10
#      print(triple(5))   # 50 — lambda reads multiplier at call time!
#
#   3. The retry pattern (no real API call needed here):
#      def retry_demo(fn, retries=3):
#          for i in range(1, retries + 1):
#              try:
#                  return fn()     # fn is called with no arguments
#              except Exception as e:
#                  print(f"Attempt {i} failed: {e}")
#                  if i == retries: raise
#
#      # Simulate a flaky function that fails twice then succeeds:
#      attempt_count = 0
#      def flaky():
#          global attempt_count
#          attempt_count += 1
#          if attempt_count < 3:
#              raise ConnectionError("Timeout")
#          return "Success!"
#
#      result = retry_demo(flaky)   # pass the function itself, no ()
#      print(result)                # "Success!"
# --- live-code below ---


---
## 7. Error Handling: `try / except / finally`

Production API code always wraps calls in `try/except`. Key rules:
- Catch the **most specific** exception type first, then broader ones
- Use `finally` for cleanup that must run regardless (e.g. closing a file)
- Re-raise with `raise` (no argument) to preserve the original traceback
- Never use bare `except:` — it catches `KeyboardInterrupt` and `SystemExit` too

In [ ]:
# ── 7a. try/except/else/finally ──────────────────────────────────────────────
# STEPS:
#   1. Basic pattern:
#      try:
#          result = int("not a number")
#      except ValueError as e:
#          print(f"Conversion failed: {e}")
#      else:
#          print(f"Got: {result}")   # only runs if no exception
#      finally:
#          print("Always runs")
#
#   2. Multiple exception types (mirrors Session 3 safe_chat()):
#      def parse_json_reply(text: str) -> dict:
#          try:
#              return json.loads(text)
#          except json.JSONDecodeError as e:
#              print(f"Model did not return valid JSON: {e}")
#              return {}        # safe fallback
#          except Exception as e:
#              print(f"Unexpected error: {e}")
#              raise            # re-raise unexpected errors
#
#      print(parse_json_reply('{"name": "Priya", "role": "engineer"}'))  # works
#      print(parse_json_reply("I am a helpful assistant."))               # JSONDecodeError
# --- live-code below ---

try:

    divide = 25/"apple"

except ZeroDivisionError:

    print('denominator cannot be zero')

except TypeError:

    print('Number cannot be divided by string')





Number cannot be divided by string


In [ ]:
# ── 7b. Custom exceptions ────────────────────────────────────────────────────
# STEPS:
#   1. Define two custom exceptions:
#      class BudgetExceededError(Exception):
#          """Raised when the session cost limit is hit."""
#
#      class MissingAPIKeyError(Exception):
#          """Raised when OPENAI_API_KEY is not set."""
#
#   2. Write a function check_budget(total_cost: float, limit: float = 0.10):
#      if total_cost > limit:
#          raise BudgetExceededError(
#              f"Cost ${total_cost:.4f} exceeds limit ${limit:.4f}. "
#              "Set MOCK=1 or increase limit."
#          )
#
#   3. Try calling it:
#      try:
#          check_budget(0.15)
#      except BudgetExceededError as e:
#          print(f"Caught: {e}")
# --- live-code below ---


---
## 8. Classes and Objects

The OpenAI SDK returns objects like `ChatCompletion`, `Choice`, and `CompletionUsage`. Understanding classes helps you read the SDK source, write duck-typed mocks (like `mock_chat_response()` in `utils/helpers.py`), and define your own data containers.

```python
class ClassName:
    class_var = "shared by all instances"

    def __init__(self, arg1, arg2):   # constructor
        self.arg1 = arg1              # instance attribute

    def method(self):                 # instance method
        return self.arg1

    def __repr__(self):               # string representation
        return f"ClassName({self.arg1!r})"
```

In [31]:
# ── 8a. Build a duck-typed mock response ─────────────────────────────────────
# GOAL: Understand how mock_chat_response() in utils/helpers.py works
#       by building a simplified version from scratch
#
# STEPS:
#   1. class Usage:
#        def __init__(self, prompt=10, completion=20):
#            self.prompt_tokens = prompt
#            self.completion_tokens = completion
#            self.total_tokens = prompt + completion
#
#   2. class Message:
#        def __init__(self, content, role="assistant"):
#            self.content = content
#            self.role = role
#
#   3. class Choice:
#        def __init__(self, content):
#            self.message = Message(content)
#            self.finish_reason = "stop"
#
#   4. class MockResponse:
#        def __init__(self, text):
#            self.choices = [Choice(text)]
#            self.usage   = Usage()
#            self.model   = "mock"
#        def __repr__(self):
#            return f"MockResponse(text={self.choices[0].message.content!r})"
#
#   5. r = MockResponse("NumPy is great for numerical computing.")
#      print(r.choices[0].message.content)   # same access pattern as the real SDK!
#      print(r.usage.total_tokens)
#      print(r)
# --- live-code below ---

class TravelPlanner():

    def __init__(self,starting_point, destination,budget,num_of_days):

        self.starting_point = starting_point
        self.destination = destination
        self.budget = budget
        self.num_of_days = num_of_days
    

    def plan_restaurents(self,veg_only=False):

        print(self.budget)

        # llms to plan based on budget

        meal_plan = f"Meal Plan For this budget {self.budget}"

        return meal_plan
    

    def travel_itenary(self):

        print('starting_point:',self.starting_point)

        print('destination:',self.destination)

        print('budget:',self.budget)

        print('Number Of Days:',self.num_of_days)

        print('Generating Meal Plan:',self.plan_restaurents(veg_only=False))


        print('travel iternary generated')


        return 'Travel Itenary plan ...'



    


In [34]:
plan1 = TravelPlanner(starting_point='chennai',destination='madurai',budget=10000,num_of_days=5)

In [35]:
plan1.plan_restaurents()

10000


'Meal Plan For this budget 10000'

In [36]:
plan1.travel_itenary()

starting_point: chennai
destination: madurai
budget: 10000
Number Of Days: 5
10000
Generating Meal Plan: Meal Plan For this budget 10000
travel iternary generated


'Travel Itenary plan ...'

In [54]:
import random

number = random.randint(10,20)

# number = 13

print(number)


if any([number%(n+1) == 0 for n in range(1,number-1)]):

    print([number%(n+1) == 0 for n in range(1,number-1)])

    print('not a prime number')

else:

    print('prime number')

13
prime number


---
## 9. JSON: `json.loads` and `json.dumps`

Structured outputs (Session 3) return JSON **strings** that you must parse. Conversely, tool call arguments arrive as JSON strings too.

```python
json.loads(string)           # string → Python dict/list
json.dumps(obj, indent=2)    # Python dict/list → formatted string
json.dumps(obj)              # compact (no spaces)
```

In [ ]:
# ── 9a. JSON round-trip — the tool call argument pattern ─────────────────────
# STEPS:
#   1. Simulate the JSON string the model sends as tool call arguments:
#      args_string = '{"amount": 10000, "gst_rate": 18}'
#
#   2. Parse it:
#      args = json.loads(args_string)
#      print(type(args), args)              # dict
#      print(args["amount"], args["gst_rate"])
#
#   3. Call a function using ** unpacking:
#      def calculate_gst(amount, gst_rate):
#          gst = round(amount * gst_rate / 100, 2)
#          return {"base": amount, "gst": gst, "total": round(amount + gst, 2)}
#
#      result = calculate_gst(**args)       # same as calculate_gst(amount=10000, gst_rate=18)
#      print(result)
#
#   4. Serialise the result back to a string (to send as the tool response):
#      result_str = json.dumps(result)
#      print(repr(result_str))              # this is what goes in the "content" of role="tool"
#
#   5. Pretty-print for debugging:
#      print(json.dumps(result, indent=2))
# --- live-code below ---


In [ ]:
# ── 9b. Handling malformed JSON from the model ───────────────────────────────
# STEPS:
#   1. Write safe_json_parse(text: str, fallback=None):
#      try:
#          return json.loads(text)
#      except json.JSONDecodeError:
#          print(f"[warn] Could not parse JSON: {text[:80]!r}")
#          return fallback
#
#   2. Test with valid JSON:
#      print(safe_json_parse('{"status": "ok"}'))       # {'status': 'ok'}
#
#   3. Test with model preamble before the JSON (a common failure mode):
#      bad = "Here is the JSON:\n{\"status\": \"ok\"}"  # model added text before JSON
#      print(safe_json_parse(bad, fallback={}))
#
#   4. Fix: strip markdown fences and leading text:
#      import re
#      cleaned = re.sub(r"^[^{\[]*", "", bad.strip())   # drop chars before first { or [
#      print(safe_json_parse(cleaned))
# --- live-code below ---


---
## 10. File I/O with `pathlib`

`pathlib.Path` is the modern way to work with file paths — cross-platform, composable, and readable. In AI apps you'll use it to load prompt templates, save outputs, and locate the project root (as in the setup cell of every session notebook).

```python
from pathlib import Path

p = Path("data/invoice.txt")
text = p.read_text(encoding="utf-8")     # read a file in one line
p.write_text("hello", encoding="utf-8")  # write a file in one line
p.exists()     # bool
p.parent       # parent directory as a Path
p.stem         # filename without extension
p.suffix       # extension including dot, e.g. '.txt'
p / "subdir" / "file.txt"   # path joining with /
```

In [57]:
# ── 10a. Read a file and use its content as a prompt ─────────────────────────
# STEPS:
#   1. Locate the project root (same pattern as session setup cells):
#      root = Path(os.getcwd())
#      for _ in range(5):
#          if (root / "utils").is_dir(): break
#          root = root.parent
#      print("Project root:", root)
#
#   2. List all .ipynb files under notebooks/:
#      notebooks = sorted(root.glob("notebooks/**/*.ipynb"))
#      for nb in notebooks:
#          print(nb.relative_to(root))
#
#   3. Read .env.example and print its contents:
#      env_example = (root / ".env.example").read_text()
#      print(env_example)
#
#   4. Write a small prompt template to data/ and read it back:
#      data_dir = root / "data"
#      data_dir.mkdir(exist_ok=True)
#      template_path = data_dir / "prompt_template.txt"
#      template_path.write_text("You are a {role}. Answer in {language}.")
#      template = template_path.read_text()
#      filled = template.format(role="financial advisor", language="simple English")
#      print(filled)
# --- live-code below ---


import os


print(os.getcwd())


print(os.listdir())


/home/chandrakanth/msme_courses/python_openai_2026_06/notebooks/weekend2
['00_python_basics.ipynb', '03_advanced_conversational.ipynb']


In [58]:
os.mkdir('log_files')

In [59]:
with open('log_files/file_001.txt','w') as file:

    file.write("log from openai llm : openai-4.5-mini is called")

    file.close()

---
## Exercises

> **Exercise 1 — Message History Manager**
>
> Write a class `ConversationHistory` that wraps the `messages` list:
> - `__init__(self, system: str)` — initialises with the system message
> - `add_user(self, text: str)` — appends a user message
> - `add_assistant(self, text: str)` — appends an assistant message
> - `trim(self, keep_last_n: int)` — keeps the system message + the last N messages
> - `token_estimate(self) -> int` — returns `sum(len(m["content"]) for m in self.messages) // 4`
> - `__len__(self)` — returns `len(self.messages)`
> - `__repr__(self)` — e.g. `"ConversationHistory(4 messages, ~32 tokens)"`
>
> Test it with a 5-turn conversation, then call `trim(4)` and verify the history shrank.

In [ ]:
# your turn:


> **Exercise 2 — Safe JSON Extractor with Schema Validation**
>
> Write `extract_json(text: str, required_keys: list[str]) -> dict`:
> 1. Strip any markdown code fences (` ```json ... ``` `) from `text` before parsing.
> 2. Parse with `json.loads()` inside a `try/except json.JSONDecodeError`.
> 3. After parsing, verify all `required_keys` are present; if not, raise a `ValueError`.
> 4. Return the parsed dict.
>
> Test cases:
> - Valid: `` "```json\n{\"name\": \"Priya\", \"role\": \"engineer\"}\n```" ``
> - Invalid JSON: `"Here is the data: name=Priya, role=engineer"`
> - Missing key: `'{"name": "Priya"}'` with `required_keys=["name", "role"]`

In [ ]:
# your turn:


---
## Recap

- **f-strings** with format specs (`:,`, `:.6f`, `!r`) handle all prompt and cost formatting cleanly.
- The `messages` list is a **list of dicts** — fluency with dict access (`.get()`, `**unpacking`, comprehensions) is essential.
- `**kwargs` lets you forward arbitrary parameters through helper functions without listing them all — used in every `chat_turn()` wrapper.
- `lambda: fn()` creates a zero-argument callable that captures the current scope — the pattern used in `retry()`.
- Always `json.loads(tool_call.function.arguments)` — arguments arrive as a JSON **string**, not a dict.
- `pathlib.Path` for all file paths — readable, cross-platform, and composable with `/`.
- Classes with `__repr__` make debugging vastly easier; duck-typing lets mock objects stand in for real SDK responses.